# 04 — Airy Diffraction Pattern

Compute the theoretical `[2J₁(u)/u]²` diffraction pattern for:
- a filled circular aperture,
- an annular aperture (with subreflector blockage),

and compare against the Gaussian approximation.

**No external data needed.**

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from beamlab21.compute import compute_airy_pattern
from beamlab21.models import twoD_Gaussian

## 1 · Parameters

In [ ]:
FREQ_MHZ  = 400.0   # MHz
D_APT     = 6.0     # aperture diameter [m]
D_BLOCK   = 1.2     # subreflector blockage diameter [m] (≈ 20 % of D)
N_PIX     = 256
XY_MAX    = 15.0    # half-window [deg]

x = np.linspace(-XY_MAX, XY_MAX, N_PIX)
y = np.linspace(-XY_MAX, XY_MAX, N_PIX)

lam_m = 299.792458 / FREQ_MHZ      # wavelength [m]
print(f"λ = {lam_m*100:.1f} cm  at {FREQ_MHZ:.0f} MHz")
print(f"D/λ = {D_APT/lam_m:.1f}  →  first null ≈ {np.rad2deg(1.22*lam_m/D_APT):.2f}°")

## 2 · Compute patterns

In [ ]:
# Filled circular aperture
airy_full = compute_airy_pattern(
    freq_mhz=FREQ_MHZ, aperture_diam_m=D_APT,
    x=x, y=y,
)

# Annular aperture (with blockage)
airy_ann = compute_airy_pattern(
    freq_mhz=FREQ_MHZ, aperture_diam_m=D_APT,
    x=x, y=y, blockage_diam_m=D_BLOCK,
)

# Gaussian approximation: σ ≈ 0.45 · λ/D  (standard illumination factor)
sigma_gauss = np.rad2deg(0.45 * lam_m / D_APT)
gauss_approx = twoD_Gaussian(x, y, [1.0, sigma_gauss, sigma_gauss, 0.0, 0.0, 0.0])

print(f"Airy (full)  peak = {airy_full.max():.4f}")
print(f"Airy (ann.)  peak = {airy_ann.max():.4f}")
print(f"Gaussian σ = {sigma_gauss:.3f}°")

## 3 · 2D maps

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
kw = dict(cmap="inferno", vmin=0, vmax=1)

titles = ["Airy (filled aperture)", f"Airy (annular, ε={D_BLOCK/D_APT:.2f})",
          f"Gaussian approx (σ={sigma_gauss:.2f}°)"]
maps   = [airy_full / airy_full.max(), airy_ann / airy_ann.max(),
          gauss_approx / gauss_approx.max()]

for ax, img, title in zip(axes, maps, titles):
    im = ax.pcolormesh(x, y, img, **kw)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("x [deg]")
    ax.set_ylabel("y [deg]")
plt.colorbar(im, ax=axes[-1], label="Normalised intensity")
plt.suptitle(f"Diffraction patterns at {FREQ_MHZ:.0f} MHz,  D={D_APT} m",
             fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

## 4 · 1D radial cuts (log scale) — sidelobe structure

In [ ]:
mid  = N_PIX // 2
r    = x[mid:]   # positive half

cut_full  = airy_full[mid, mid:]  / airy_full.max()
cut_ann   = airy_ann[mid,  mid:]  / airy_ann.max()
cut_gauss = gauss_approx[mid, mid:] / gauss_approx.max()

fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogy(r, cut_full,  label="Airy (filled)")
ax.semilogy(r, cut_ann,   label=f"Airy (annular, ε={D_BLOCK/D_APT:.2f})", linestyle="--")
ax.semilogy(r, cut_gauss, label="Gaussian approx",   linestyle=":")

# Mark the first Airy null (1.22 λ/D)
first_null = np.rad2deg(1.22 * lam_m / D_APT)
ax.axvline(first_null, color="k", linewidth=0.8, linestyle="-.",
           label=f"1st null = {first_null:.2f}°")

ax.set_xlim(0, XY_MAX)
ax.set_ylim(1e-5, 2)
ax.set_xlabel("θ [deg]")
ax.set_ylabel("Normalised intensity (log scale)")
ax.set_title("Radial cuts through beam centre")
ax.legend()
plt.tight_layout()
plt.show()

## 5 · Frequency dependence — beam narrows as ν increases

In [ ]:
freqs = [300, 400, 600, 1000]

fig, ax = plt.subplots(figsize=(8, 4))
for f in freqs:
    airy_f = compute_airy_pattern(f, D_APT, x, y)
    cut    = airy_f[mid, mid:] / airy_f.max()
    ax.plot(r, cut, label=f"{f} MHz")

ax.set_yscale("log")
ax.set_xlim(0, XY_MAX)
ax.set_ylim(1e-5, 2)
ax.set_xlabel("θ [deg]")
ax.set_ylabel("Normalised intensity")
ax.set_title(f"Airy pattern at multiple frequencies  (D={D_APT} m)")
ax.legend()
plt.tight_layout()
plt.show()